# 01 Prepare Data

Download and verify CIFAR-10 and Tiny ImageNet.

In [ ]:

from pathlib import Path
import os, json, yaml, math, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader, random_split, Subset

from src.utils.seed import set_seed
from src.models.unet import UNetModel
from src.training.trainer import SDDTrainer
from src.evaluation.feature_extract import extract_features
from src.evaluation.linear_probe import train_linear_probe
from src.evaluation.fid import FIDEvaluator

ROOT = Path('.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(42)


In [ ]:

data_root = ROOT / "data"
data_root.mkdir(exist_ok=True)

cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

cifar_train = datasets.CIFAR10(root=str(data_root), train=True, download=True, transform=cifar_transform)
cifar_test = datasets.CIFAR10(root=str(data_root), train=False, download=True, transform=cifar_transform)
print("CIFAR-10:", len(cifar_train), len(cifar_test))

# Tiny ImageNet download helper
tiny_root = data_root / "tiny-imagenet-200"
tiny_root.mkdir(exist_ok=True)

tiny_zip = data_root / "tiny-imagenet-200.zip"
url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

print("Tiny ImageNet expected layout:")
print(tiny_root / "train")
print(tiny_root / "val")
print("If not present, download and extract manually or use the helper cell below.")


In [ ]:

# Optional: download Tiny ImageNet if internet is available in your environment.
# Uncomment to run.
# import urllib.request, zipfile
# if not tiny_zip.exists():
#     urllib.request.urlretrieve(url, tiny_zip)
# with zipfile.ZipFile(tiny_zip, 'r') as zf:
#     zf.extractall(data_root)
